In [4]:
states = ["phi", "chi1", "chi2", "chi3", "phi_prime", "chi1_prime", "chi2_prime", "chi3_prime"]

In [5]:
import numpy as np

def unpack_state(states):
    states = np.asarray(states)

    n_k = int(len(states) / 2) - 1

    phi = states[0]
    chi = states[1 : 1 + n_k]

    phi_prime = states[1 + n_k]
    chi_prime = states[2 + n_k : 2 + 2*n_k]

    return phi, phi_prime, chi, chi_prime


In [6]:
phi, phi_prime, chi, chi_prime = unpack_state(states)
print(f"phi: {phi}")
print(f"phi_prime: {phi_prime}")
print(f"chi: {chi}")
print(f"chi_prime: {chi_prime}")

phi: phi
phi_prime: phi_prime
chi: ['chi1' 'chi2' 'chi3']
chi_prime: ['chi1_prime' 'chi2_prime' 'chi3_prime']


In [7]:
import numpy as np

def make_k_array(k_min, k_max, n_k):
    """
    Returns n_k equally spaced k-modes between k_min and k_max (inclusive).
    """
    return np.linspace(k_min, k_max, n_k)


In [12]:
print(make_k_array(1.0, 10, 20))

[ 1.          1.47368421  1.94736842  2.42105263  2.89473684  3.36842105
  3.84210526  4.31578947  4.78947368  5.26315789  5.73684211  6.21052632
  6.68421053  7.15789474  7.63157895  8.10526316  8.57894737  9.05263158
  9.52631579 10.        ]


In [17]:
n_k = 4

header = (
    ["t", "phi", "phi_prime"]
    + [f"chi_{i}" for i in range(n_k)]
    + [f"chi_dot_{j}" for j in range(n_k)]
    + ["a"]
)
print(header)

['t', 'phi', 'phi_prime', 'chi_0', 'chi_1', 'chi_2', 'chi_3', 'chi_dot_0', 'chi_dot_1', 'chi_dot_2', 'chi_dot_3', 'a']


In [18]:
import numpy as np

def H_from_phi_of_N(N, phi, phi_prime, V_func, Mpl=1.0, t0=0.0):
    """
    Given arrays phi(N) and phi'(N), compute:
      - H(N) from Friedmann,
      - cosmic time t(N) via dt/dN = 1/H,
      - and return H(t) as arrays (t, H).

    Assumes a single canonical scalar field and reduced Planck mass Mpl (set Mpl=1 if using Planck units).
    """
    N = np.asarray(N, dtype=float)
    phi = np.asarray(phi, dtype=float)
    phi_prime = np.asarray(phi_prime, dtype=float)

    if not (N.shape == phi.shape == phi_prime.shape):
        raise ValueError("N, phi, phi_prime must have the same shape.")

    # Potential along the trajectory
    V = V_func(phi)

    # Friedmann in N-variable:
    # 3 Mpl^2 H^2 = V + (1/2) (dphi/dt)^2, and dphi/dt = H * dphi/dN
    # => H^2 = V / (3 Mpl^2 - (1/2) phi'^2)
    denom = (3.0 * Mpl**2) - 0.5 * phi_prime**2
    if np.any(denom <= 0):
        bad = np.where(denom <= 0)[0][:10]
        raise ValueError(
            "Denominator 3 Mpl^2 - 0.5 phi'^2 must be > 0. "
            f"Failed at indices {bad.tolist()} (example denom={denom[bad[0]]})."
        )

    H = np.sqrt(V / denom)

    # Build cosmic time: dt/dN = 1/H
    invH = 1.0 / H
    t = np.empty_like(N)
    t[0] = t0
    t[1:] = t0 + np.cumsum(0.5 * (invH[1:] + invH[:-1]) * (N[1:] - N[:-1]))  # trapezoid

    # Ensure (t, H) is sorted by increasing t (in case N decreases)
    order = np.argsort(t)
    t = t[order]
    H = H[order]

    return t, H


def H_of_t_interpolator(t, H):
    """
    Returns a callable H_of_t(t_query) using simple linear interpolation.
    """
    t = np.asarray(t, dtype=float)
    H = np.asarray(H, dtype=float)
    if t.ndim != 1 or H.ndim != 1 or t.size != H.size:
        raise ValueError("t and H must be 1D arrays of the same length.")
    if np.any(np.diff(t) <= 0):
        raise ValueError("t must be strictly increasing for interpolation.")

    def H_of_t(t_query):
        return np.interp(t_query, t, H)

    return H_of_t


# ---------------- Example usage ----------------
if __name__ == "__main__":
    # Replace this with YOUR potential function V(phi)
    def V_func(phi):
        # example: V = 1/2 m^2 phi^2
        m = 1e-5
        return 0.5 * m**2 * phi**2

    # Example dummy data (replace with your arrays)
    N = np.linspace(0.0, 10.0, 2001)
    phi = 10.0 - 0.1 * N
    phi_prime = np.gradient(phi, N)  # or use your known phi'(N)

    t, H = H_from_phi_of_N(N, phi, phi_prime, V_func, Mpl=1.0, t0=0.0)

    # Build H(t) callable
    H_of_t = H_of_t_interpolator(t, H)

    # Example query
    tq = np.linspace(t[0], t[-1], 100)
    Hq = H_of_t(tq)

    print("t range:", t[0], "to", t[-1])
    print("H(t[0]), H(t[-1]):", H[0], H[-1])


t range: 0.0 to 257864.34658010417
H(t[0]), H(t[-1]): 4.085889232227193e-05 3.677300309004475e-05
